# Credit Card Fraud Detection

This notebook follows the CS770 project proposal using three distinct fraud-detection dataset sources when the files are available:

- `fraudTrain.csv` plus `fraudTest.csv` (one transaction dataset with an official split)
- `creditcard.csv`
- IEEE-CIS Fraud Detection (`train_transaction.csv`, with optional `train_identity.csv`)

The goal is binary classification: predict whether a credit card transaction is fraud or not fraud.


## Project Plan

The proposal says to compare Random Forest and Logistic Regression, account for class imbalance, use dimensionality reduction with PCA, and report accuracy, precision, recall, F1-score, confusion matrix, ROC curve, and AUC.

Because the dataset sources have different schemas, this notebook trains and evaluates separate models for each source. The final section combines their metrics into one comparison table while keeping the source dataset label visible.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 67
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid")


## 1. Load the Selected Datasets


In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_CANDIDATES = [PROJECT_ROOT / "datasets", PROJECT_ROOT.parent / "datasets"]
DATA_DIR = next(
    (
        path
        for path in DATA_CANDIDATES
        if (path / "creditcard.csv").exists()
        and (path / "fraudTrain.csv").exists()
        and (path / "fraudTest.csv").exists()
    ),
    DATA_CANDIDATES[0],
)

CREDITCARD_PATH = DATA_DIR / "creditcard.csv"
TRAIN_PATH = DATA_DIR / "fraudTrain.csv"
TEST_PATH = DATA_DIR / "fraudTest.csv"
IEEE_TRANSACTION_PATH = DATA_DIR / "train_transaction.csv"
IEEE_IDENTITY_PATH = DATA_DIR / "train_identity.csv"
HAS_IEEE_CIS = IEEE_TRANSACTION_PATH.exists()

print("Creditcard data path:", CREDITCARD_PATH)
print("Training data path:", TRAIN_PATH)
print("Testing data path:", TEST_PATH)
print("IEEE-CIS transaction path:", IEEE_TRANSACTION_PATH)
print("IEEE-CIS available:", HAS_IEEE_CIS)

creditcard_df = pd.read_csv(CREDITCARD_PATH)
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
ieee_transaction_df = pd.read_csv(IEEE_TRANSACTION_PATH) if HAS_IEEE_CIS else None

print("creditcard shape:", creditcard_df.shape)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
if ieee_transaction_df is not None:
    print("IEEE-CIS transaction shape:", ieee_transaction_df.shape)
else:
    print("IEEE-CIS files not found. Add train_transaction.csv to the datasets folder to run Section 10.")

train_df.head()


In [ ]:
print("fraudTrain/fraudTest schema")
train_df.info()

print("\ncreditcard.csv schema")
creditcard_df.info()

if ieee_transaction_df is not None:
    print("\nIEEE-CIS train_transaction.csv schema")
    ieee_transaction_df.info()


## 2. Initial Analysis: Class Imbalance

Fraud detection datasets are usually highly imbalanced, meaning most transactions are not fraud. Because of that, accuracy alone can be misleading. Precision, recall, F1-score, PR-AUC, ROC-AUC, and the confusion matrix are more useful.


In [ ]:
def show_class_balance(df, label="dataset", target_column="is_fraud"):
    counts = df[target_column].value_counts().sort_index()
    percents = df[target_column].value_counts(normalize=True).sort_index() * 100
    balance = pd.DataFrame({"count": counts, "percent": percents.round(4)})
    balance.index = ["not fraud", "fraud"]
    print(label)
    display(balance)
    return balance

train_balance = show_class_balance(train_df, "fraudTrain.csv", target_column="is_fraud")
test_balance = show_class_balance(test_df, "fraudTest.csv", target_column="is_fraud")
creditcard_balance = show_class_balance(creditcard_df, "creditcard.csv", target_column="Class")
if ieee_transaction_df is not None:
    ieee_balance = show_class_balance(ieee_transaction_df, "IEEE-CIS train_transaction.csv", target_column="isFraud")


In [ ]:
plot_specs = [
    (train_df, "is_fraud", "fraudTrain.csv"),
    (test_df, "is_fraud", "fraudTest.csv"),
    (creditcard_df, "Class", "creditcard.csv"),
]
if ieee_transaction_df is not None:
    plot_specs.append((ieee_transaction_df, "isFraud", "IEEE-CIS"))

fig, axes = plt.subplots(1, len(plot_specs), figsize=(5 * len(plot_specs), 4))
if len(plot_specs) == 1:
    axes = [axes]

for ax, (df, target, title) in zip(axes, plot_specs):
    sns.countplot(data=df, x=target, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("fraud label")
    ax.set_ylabel("transactions")
    ax.set_xticklabels(["not fraud", "fraud"])

plt.tight_layout()
plt.show()


## 3. Experiment 1: `fraudTrain.csv` and `fraudTest.csv`

The original transaction dataset contains identifiers and personally identifying text fields that are not good model inputs. This section keeps useful transaction features and creates date-based features like transaction hour, day of week, and cardholder age.


In [ ]:
def add_time_features(df):
    df = df.copy()
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

    df["transaction_hour"] = df["trans_date_trans_time"].dt.hour
    df["transaction_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek
    df["transaction_month"] = df["trans_date_trans_time"].dt.month
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25

    return df

train_features_df = add_time_features(train_df)
test_features_df = add_time_features(test_df)

feature_columns = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_month",
    "age",
    "category",
    "gender",
]

target_column = "is_fraud"

X_train_full = train_features_df[feature_columns]
y_train_full = train_features_df[target_column]
X_test_full = test_features_df[feature_columns]
y_test_full = test_features_df[target_column]

X_train_full.head()


## 4. Sampling and Preprocessing

The full `fraudTrain.csv` and `fraudTest.csv` files are large enough to make repeated model experiments slow. The stratified sample keeps the class ratio intact while making the notebook practical to rerun.


In [ ]:
USE_SAMPLE = True
TRAIN_SAMPLE_SIZE = 100_000
TEST_SAMPLE_SIZE = 50_000


def stratified_sample(X, y, sample_size, random_state=RANDOM_STATE):
    if (not USE_SAMPLE) or len(X) <= sample_size:
        return X.copy(), y.copy()

    _, X_sample, _, y_sample = train_test_split(
        X,
        y,
        test_size=sample_size,
        stratify=y,
        random_state=random_state,
    )
    return X_sample.reset_index(drop=True), y_sample.reset_index(drop=True)


X_train, y_train = stratified_sample(X_train_full, y_train_full, TRAIN_SAMPLE_SIZE)
X_test, y_test = stratified_sample(X_test_full, y_test_full, TEST_SAMPLE_SIZE)

print("training shape:", X_train.shape)
print("testing shape:", X_test.shape)
show_class_balance(pd.DataFrame({target_column: y_train}), "Model training set")
show_class_balance(pd.DataFrame({target_column: y_test}), "Model testing set")


In [ ]:
def build_one_hot_encoder():
    return OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)


numeric_features = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_month",
    "age",
]

categorical_features = ["category", "gender"]


def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", build_one_hot_encoder()),
                ]),
                categorical_features,
            ),
        ],
        remainder="drop",
    )


def build_pca():
    return PCA(n_components=0.95, random_state=RANDOM_STATE)


## 5. Baseline Models and Evaluation


In [ ]:
logistic_regression = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("pca", build_pca()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
    )),
])

random_forest = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("pca", build_pca()),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])

models = {
    "Logistic Regression": logistic_regression,
    "Random Forest": random_forest,
}


In [ ]:
fitted_models = {}
for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    print(f"Finished {model_name}")


In [ ]:
for model_name, model in fitted_models.items():
    n_components = model.named_steps["pca"].n_components_
    explained_variance = model.named_steps["pca"].explained_variance_ratio_.sum()
    print(f"{model_name}: PCA kept {n_components} components explaining {explained_variance:.2%} variance")


In [ ]:
def evaluate_model(model_name, model, X_test, y_test, threshold=0.50, dataset=None, evaluation="default threshold"):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "dataset": dataset,
        "evaluation": evaluation,
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
    }

    print(f"\n{model_name} ({evaluation}, threshold={threshold:.2f})")
    print(classification_report(y_test, y_pred, target_names=["not fraud", "fraud"], zero_division=0))

    return metrics, y_pred, y_prob


results = []
predictions = {}

for model_name, model in fitted_models.items():
    metrics, y_pred, y_prob = evaluate_model(
        model_name,
        model,
        X_test,
        y_test,
        dataset="fraudTrain/fraudTest",
        evaluation="default threshold",
    )
    results.append(metrics)
    predictions[model_name] = {"y_pred": y_pred, "y_prob": y_prob}

results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
results_df


The baseline table separates ranking metrics such as ROC-AUC and PR-AUC from threshold-dependent metrics such as precision, recall, and F1. Because fraud is rare, PR-AUC and recall are especially important when judging whether the model is actually useful for finding fraud cases.


## 6. Hyperparameter Tuning and Feature Importance

This section uses the `GridSearchCV` result instead of leaving it as dead code. It evaluates the best estimator on the held-out test split and adds it to the same results table as the baseline models.


In [ ]:
rf_grid = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("pca", build_pca()),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 5],
}
grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)
print("best cross-validation ROC-AUC:", grid_search.best_score_)
print("best params:", grid_search.best_params_)

grid_metrics, grid_y_pred, grid_y_prob = evaluate_model(
    "Tuned Random Forest (GridSearchCV)",
    grid_search.best_estimator_,
    X_test,
    y_test,
    dataset="fraudTrain/fraudTest",
    evaluation="grid search best estimator",
)
results.append(grid_metrics)
results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False)

fitted_models["Tuned Random Forest (GridSearchCV)"] = grid_search.best_estimator_
predictions["Tuned Random Forest (GridSearchCV)"] = {"y_pred": grid_y_pred, "y_prob": grid_y_prob}

results_df


In [ ]:
rf_feature_importance_model = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
])
rf_feature_importance_model.fit(X_train, y_train)

feature_names = rf_feature_importance_model.named_steps["preprocess"].get_feature_names_out()
feature_names = [name.replace("numeric__", "").replace("categorical__", "") for name in feature_names]
importances = rf_feature_importance_model.named_steps["model"].feature_importances_

importance_df = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
    .head(15)
)

display(importance_df)

fig, ax = plt.subplots(figsize=(9, 6))
sns.barplot(data=importance_df, x="importance", y="feature", ax=ax, color="steelblue")
ax.set_title("Top Random Forest Feature Importances")
ax.set_xlabel("importance")
ax.set_ylabel("feature")
plt.tight_layout()
plt.show()


The feature-importance model is trained without PCA so the importances map back to original engineered features rather than anonymous principal components. This makes the plot easier to discuss in the report and complements the PCA experiment instead of duplicating it.


## 7. Threshold Tuning

Fraud detection usually cares about catching fraud cases without creating too many false alarms. The default threshold of 0.50 is not always best, so this section compares thresholds using precision, recall, and F1-score, then re-evaluates each model at its best F1 threshold.


In [ ]:
threshold_rows = []
thresholds = np.arange(0.05, 0.96, 0.05)

for model_name, preds in predictions.items():
    y_prob = preds["y_prob"]
    for threshold in thresholds:
        y_pred_threshold = (y_prob >= threshold).astype(int)
        threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(y_test, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_test, y_pred_threshold, zero_division=0),
            "f1": f1_score(y_test, y_pred_threshold, zero_division=0),
        })

threshold_df = pd.DataFrame(threshold_rows)
best_thresholds_df = (
    threshold_df.sort_values(["model", "f1"], ascending=[True, False])
    .drop_duplicates("model")
    .sort_values("f1", ascending=False)
)

display(best_thresholds_df)
threshold_df.sort_values(["model", "f1"], ascending=[True, False]).groupby("model").head(5)


In [ ]:
fig, axes = plt.subplots(1, len(fitted_models), figsize=(6 * len(fitted_models), 4), sharey=True)
if len(fitted_models) == 1:
    axes = [axes]

for ax, model_name in zip(axes, fitted_models.keys()):
    subset = threshold_df[threshold_df["model"] == model_name]
    ax.plot(subset["threshold"], subset["precision"], label="precision")
    ax.plot(subset["threshold"], subset["recall"], label="recall")
    ax.plot(subset["threshold"], subset["f1"], label="f1")
    ax.set_title(model_name)
    ax.set_xlabel("threshold")
    ax.set_ylabel("score")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
tuned_results = []
tuned_predictions = {}

for row in best_thresholds_df.itertuples(index=False):
    model = fitted_models[row.model]
    metrics, y_pred, y_prob = evaluate_model(
        row.model,
        model,
        X_test,
        y_test,
        threshold=row.threshold,
        dataset="fraudTrain/fraudTest",
        evaluation="best F1 threshold",
    )
    tuned_results.append(metrics)
    tuned_predictions[row.model] = {"y_pred": y_pred, "y_prob": y_prob}

tuned_results_df = pd.DataFrame(tuned_results).sort_values("f1", ascending=False)
tuned_results_df


In [ ]:
fig, axes = plt.subplots(1, len(tuned_predictions), figsize=(6 * len(tuned_predictions), 5))
if len(tuned_predictions) == 1:
    axes = [axes]

for ax, (model_name, preds) in zip(axes, tuned_predictions.items()):
    cm = confusion_matrix(y_test, preds["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not fraud", "fraud"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    threshold = tuned_results_df.loc[tuned_results_df["model"] == model_name, "threshold"].iloc[0]
    ax.set_title(f"{model_name}\nthreshold={threshold:.2f}")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=model_name)
ax.set_title("ROC Curve: fraudTrain/fraudTest")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_models.items():
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax, name=model_name)
ax.set_title("Precision-Recall Curve: fraudTrain/fraudTest")
plt.tight_layout()
plt.show()


The tuned-threshold confusion matrices show the practical tradeoff more clearly than the 0.50 default. Lower thresholds usually increase recall, which catches more fraud, but they can also reduce precision by flagging more legitimate transactions.


## 8. Experiment 2: `creditcard.csv`

`creditcard.csv` has a different structure from the `fraudTrain.csv` and `fraudTest.csv` files. It already contains mostly numeric PCA-transformed features named `V1` through `V28`, plus `Time`, `Amount`, and the target column `Class`.

Because this file does not come with a separate test CSV, this section creates a stratified train/test split from `creditcard.csv`.


In [ ]:
creditcard_target = "Class"
creditcard_feature_columns = [column for column in creditcard_df.columns if column != creditcard_target]

X_creditcard = creditcard_df[creditcard_feature_columns]
y_creditcard = creditcard_df[creditcard_target]

X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_creditcard,
    y_creditcard,
    test_size=0.20,
    stratify=y_creditcard,
    random_state=RANDOM_STATE,
)

print("creditcard training shape:", X_credit_train.shape)
print("creditcard testing shape:", X_credit_test.shape)
show_class_balance(pd.DataFrame({creditcard_target: y_credit_train}), "creditcard training split", target_column=creditcard_target)
show_class_balance(pd.DataFrame({creditcard_target: y_credit_test}), "creditcard testing split", target_column=creditcard_target)


## 9. Creditcard Preprocessing and Results

The `V1` through `V28` columns are already PCA-transformed in the public dataset, so this experiment does not apply PCA again. The pipeline imputes missing values, scales the numeric columns, and trains the same two model types.


In [ ]:
def build_creditcard_pipeline(model):
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


creditcard_models = {
    "Creditcard Logistic Regression": build_creditcard_pipeline(LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
    )),
    "Creditcard Random Forest": build_creditcard_pipeline(RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
}

fitted_creditcard_models = {}
for model_name, model in creditcard_models.items():
    print(f"Training {model_name}...")
    model.fit(X_credit_train, y_credit_train)
    fitted_creditcard_models[model_name] = model
    print(f"Finished {model_name}")


In [ ]:
creditcard_results = []
creditcard_predictions = {}

for model_name, model in fitted_creditcard_models.items():
    metrics, y_pred, y_prob = evaluate_model(
        model_name,
        model,
        X_credit_test,
        y_credit_test,
        dataset="creditcard.csv",
        evaluation="default threshold",
    )
    creditcard_results.append(metrics)
    creditcard_predictions[model_name] = {"y_pred": y_pred, "y_prob": y_prob}

creditcard_results_df = pd.DataFrame(creditcard_results).sort_values("pr_auc", ascending=False)
creditcard_results_df


In [ ]:
creditcard_threshold_rows = []
for model_name, preds in creditcard_predictions.items():
    y_prob = preds["y_prob"]
    for threshold in thresholds:
        y_pred_threshold = (y_prob >= threshold).astype(int)
        creditcard_threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(y_credit_test, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_credit_test, y_pred_threshold, zero_division=0),
            "f1": f1_score(y_credit_test, y_pred_threshold, zero_division=0),
        })

creditcard_threshold_df = pd.DataFrame(creditcard_threshold_rows)
creditcard_best_thresholds_df = (
    creditcard_threshold_df.sort_values(["model", "f1"], ascending=[True, False])
    .drop_duplicates("model")
    .sort_values("f1", ascending=False)
)

display(creditcard_best_thresholds_df)

creditcard_tuned_results = []
creditcard_tuned_predictions = {}
for row in creditcard_best_thresholds_df.itertuples(index=False):
    model = fitted_creditcard_models[row.model]
    metrics, y_pred, y_prob = evaluate_model(
        row.model,
        model,
        X_credit_test,
        y_credit_test,
        threshold=row.threshold,
        dataset="creditcard.csv",
        evaluation="best F1 threshold",
    )
    creditcard_tuned_results.append(metrics)
    creditcard_tuned_predictions[row.model] = {"y_pred": y_pred, "y_prob": y_prob}

creditcard_tuned_results_df = pd.DataFrame(creditcard_tuned_results).sort_values("f1", ascending=False)
creditcard_tuned_results_df


In [ ]:
fig, axes = plt.subplots(1, len(creditcard_tuned_predictions), figsize=(6 * len(creditcard_tuned_predictions), 5))
if len(creditcard_tuned_predictions) == 1:
    axes = [axes]

for ax, (model_name, preds) in zip(axes, creditcard_tuned_predictions.items()):
    cm = confusion_matrix(y_credit_test, preds["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not fraud", "fraud"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    threshold = creditcard_tuned_results_df.loc[creditcard_tuned_results_df["model"] == model_name, "threshold"].iloc[0]
    ax.set_title(f"{model_name}\nthreshold={threshold:.2f}")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_creditcard_models.items():
    RocCurveDisplay.from_estimator(model, X_credit_test, y_credit_test, ax=ax, name=model_name)
ax.set_title("creditcard.csv ROC Curve")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_creditcard_models.items():
    PrecisionRecallDisplay.from_estimator(model, X_credit_test, y_credit_test, ax=ax, name=model_name)
ax.set_title("creditcard.csv Precision-Recall Curve")
plt.tight_layout()
plt.show()


For `creditcard.csv`, the PCA step is intentionally skipped because the source features are already anonymized principal components. Any performance difference here is more likely driven by the model choice and threshold than by additional dimensionality reduction.


## 10. Experiment 3: IEEE-CIS Fraud Detection

This third dataset section uses the IEEE-CIS `train_transaction.csv` file when it is present in the `datasets` folder. The notebook keeps the section optional because the IEEE-CIS files are large and may not be included in every local copy of the project.


In [ ]:
def select_existing_columns(df, columns):
    return [column for column in columns if column in df.columns]


if ieee_transaction_df is not None:
    ieee_target = "isFraud"
    ieee_candidate_features = select_existing_columns(
        ieee_transaction_df,
        [
            "TransactionDT",
            "TransactionAmt",
            "ProductCD",
            "card1",
            "card2",
            "card3",
            "card4",
            "card5",
            "card6",
            "addr1",
            "addr2",
            "dist1",
            "dist2",
            "P_emaildomain",
            "R_emaildomain",
            *[f"C{i}" for i in range(1, 15)],
            *[f"D{i}" for i in range(1, 16)],
            *[f"M{i}" for i in range(1, 10)],
        ],
    )

    ieee_model_df = ieee_transaction_df[ieee_candidate_features + [ieee_target]].copy()
    X_ieee_full = ieee_model_df[ieee_candidate_features]
    y_ieee_full = ieee_model_df[ieee_target]
    X_ieee_sample, y_ieee_sample = stratified_sample(X_ieee_full, y_ieee_full, 100_000)

    X_ieee_train, X_ieee_test, y_ieee_train, y_ieee_test = train_test_split(
        X_ieee_sample,
        y_ieee_sample,
        test_size=0.20,
        stratify=y_ieee_sample,
        random_state=RANDOM_STATE,
    )

    ieee_numeric_features = X_ieee_train.select_dtypes(include=["number"]).columns.tolist()
    ieee_categorical_features = [column for column in X_ieee_train.columns if column not in ieee_numeric_features]

    ieee_preprocessor = ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                ieee_numeric_features,
            ),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", build_one_hot_encoder()),
                ]),
                ieee_categorical_features,
            ),
        ],
        remainder="drop",
    )

    ieee_models = {
        "IEEE-CIS Logistic Regression": Pipeline(steps=[
            ("preprocess", ieee_preprocessor),
            ("model", LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
                solver="lbfgs",
            )),
        ]),
        "IEEE-CIS Random Forest": Pipeline(steps=[
            ("preprocess", ieee_preprocessor),
            ("model", RandomForestClassifier(
                class_weight="balanced",
                n_estimators=200,
                max_depth=None,
                min_samples_leaf=2,
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )),
        ]),
    }

    fitted_ieee_models = {}
    ieee_results = []
    ieee_predictions = {}
    for model_name, model in ieee_models.items():
        print(f"Training {model_name}...")
        model.fit(X_ieee_train, y_ieee_train)
        fitted_ieee_models[model_name] = model
        metrics, y_pred, y_prob = evaluate_model(
            model_name,
            model,
            X_ieee_test,
            y_ieee_test,
            dataset="IEEE-CIS train_transaction.csv",
            evaluation="default threshold",
        )
        ieee_results.append(metrics)
        ieee_predictions[model_name] = {"y_pred": y_pred, "y_prob": y_prob}

    ieee_results_df = pd.DataFrame(ieee_results).sort_values("pr_auc", ascending=False)
    display(ieee_results_df)
else:
    fitted_ieee_models = {}
    ieee_predictions = {}
    ieee_results_df = pd.DataFrame()
    print("Skipping IEEE-CIS modeling because train_transaction.csv was not found in the datasets folder.")


In [ ]:
if ieee_predictions:
    ieee_threshold_rows = []
    for model_name, preds in ieee_predictions.items():
        y_prob = preds["y_prob"]
        for threshold in thresholds:
            y_pred_threshold = (y_prob >= threshold).astype(int)
            ieee_threshold_rows.append({
                "model": model_name,
                "threshold": threshold,
                "precision": precision_score(y_ieee_test, y_pred_threshold, zero_division=0),
                "recall": recall_score(y_ieee_test, y_pred_threshold, zero_division=0),
                "f1": f1_score(y_ieee_test, y_pred_threshold, zero_division=0),
            })

    ieee_threshold_df = pd.DataFrame(ieee_threshold_rows)
    ieee_best_thresholds_df = (
        ieee_threshold_df.sort_values(["model", "f1"], ascending=[True, False])
        .drop_duplicates("model")
        .sort_values("f1", ascending=False)
    )
    display(ieee_best_thresholds_df)

    ieee_tuned_results = []
    ieee_tuned_predictions = {}
    for row in ieee_best_thresholds_df.itertuples(index=False):
        model = fitted_ieee_models[row.model]
        metrics, y_pred, y_prob = evaluate_model(
            row.model,
            model,
            X_ieee_test,
            y_ieee_test,
            threshold=row.threshold,
            dataset="IEEE-CIS train_transaction.csv",
            evaluation="best F1 threshold",
        )
        ieee_tuned_results.append(metrics)
        ieee_tuned_predictions[row.model] = {"y_pred": y_pred, "y_prob": y_prob}

    ieee_tuned_results_df = pd.DataFrame(ieee_tuned_results).sort_values("f1", ascending=False)
    display(ieee_tuned_results_df)
else:
    ieee_tuned_results_df = pd.DataFrame()


In [ ]:
if fitted_ieee_models:
    fig, ax = plt.subplots(figsize=(7, 6))
    for model_name, model in fitted_ieee_models.items():
        RocCurveDisplay.from_estimator(model, X_ieee_test, y_ieee_test, ax=ax, name=model_name)
    ax.set_title("IEEE-CIS ROC Curve")
    plt.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 6))
    for model_name, model in fitted_ieee_models.items():
        PrecisionRecallDisplay.from_estimator(model, X_ieee_test, y_ieee_test, ax=ax, name=model_name)
    ax.set_title("IEEE-CIS Precision-Recall Curve")
    plt.tight_layout()
    plt.show()


The IEEE-CIS section gives the project a third distinct dataset source when the Kaggle files are available locally. Its schema is different again, so the comparison should focus on whether the same modeling approach behaves consistently across datasets rather than treating the scores as one shared benchmark.


## 11. Compare Dataset Sources

The schemas are different, so their scores should not be interpreted as one shared train/test experiment. They are related fraud-detection experiments using the same modeling approach, combined here for easier reporting.


In [ ]:
result_tables = [
    results_df,
    tuned_results_df,
    creditcard_results_df,
    creditcard_tuned_results_df,
]
if not ieee_results_df.empty:
    result_tables.append(ieee_results_df)
if not ieee_tuned_results_df.empty:
    result_tables.append(ieee_tuned_results_df)

combined_results_df = (
    pd.concat(result_tables, ignore_index=True)
    .sort_values(["dataset", "evaluation", "pr_auc"], ascending=[True, True, False])
    .reset_index(drop=True)
)

combined_results_df


In [ ]:
bar_metrics_df = combined_results_df[combined_results_df["evaluation"].isin(["default threshold", "best F1 threshold"])]

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
for ax, metric in zip(axes, ["f1", "roc_auc"]):
    sns.barplot(
        data=bar_metrics_df,
        x="dataset",
        y=metric,
        hue="model",
        ax=ax,
    )
    ax.set_title(f"{metric.upper()} by Model and Dataset")
    ax.set_xlabel("dataset")
    ax.set_ylabel(metric)
    ax.tick_params(axis="x", rotation=20)
    ax.legend(loc="lower right", fontsize="small")

plt.tight_layout()
plt.show()


The combined table and grouped bar chart are the most report-ready outputs. The table supports exact metric comparisons, while the bar chart makes it easier to describe which model and threshold strategy performed best across dataset sources.
